> **Nota de consolidação da versão atual:** os notebooks 08–13 são a linha final mais recente. O modelo campeão é Regressão Logística L2 em `tmax_enchimento`, com limiar 28,34 °C e status protótipo experimental não aprovado. A base atual usa nove municípios, treino 2001–2018 e teste 2019–2024. Resultados anteriores de Sorriso-MT com 10 safras ou limiar fixo são históricos.

# Tópico 5 e 6: Variável-Alvo e Feature Engineering
Este notebook é responsável por:
1. Criar a variável-alvo (`quebra_safra`) baseada no tendência linear passada municipal (Tópico 5).
2. Remover variáveis que causariam Target Leakage (Vazamento de Dados).
3. Criar novas variáveis preditoras baseadas nos dados agroclimáticos (Tópico 6).

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_RAIZ = Path("/content/drive/MyDrive/projeto_soja_sorriso")
except ImportError:
    # Fora do Colab, o kernel pode iniciar na raiz do repositório ou em notebooks/.
    candidatos = [Path.cwd(), *Path.cwd().parents]
    projeto_aninhado = Path.cwd() / "projeto_soja_sorriso"
    if projeto_aninhado.is_dir():
        candidatos.insert(0, projeto_aninhado)

    PASTA_RAIZ = next((p for p in candidatos
                        if (p / "notebooks").is_dir() and (p / "requirements.txt").is_file()), None)
    if PASTA_RAIZ is None:
        raise FileNotFoundError("Raiz do projeto não encontrada a partir de: " + str(Path.cwd()))

INTERIM = PASTA_RAIZ / "data" / "interim"

# Carregar os dados limpos consolidados (safras 2015 a 2024, sem 2025)
df_limpo = pd.read_csv(INTERIM / "limpo.csv")
display(df_limpo.head())

,cod_municipio,safra,area_plantada_ha,area_colhida_ha,quantidade_produzida_t,rendimento_kg_ha,rend_ref,razao_rend_ref,quebra_safra,PRECTOTCORR,...,T2M_MAX,T2M_MIN,RH2M,GWETROOT,cobertura_clima_min,amplitude_termica,precipitacao_por_temperatura,GWET_x_PREC,municipio,split
0,5105259,2001,175000.0,175000.0,557025.0,3183.0,3303.4,0.963553,0,1532.44,...,39.67,13.98,78.689835,0.746281,1.0,25.69,59.917129,1143.630843,Lucas do Rio Verde,treino
1,5105903,2001,2550.0,2550.0,6120.0,2400.0,2740.1,0.875880,1,1419.54,...,38.50,14.06,73.274752,0.395579,1.0,24.44,54.798702,561.539521,Nobres,treino
2,5106224,2001,160000.0,160000.0,489600.0,3060.0,2801.6,1.092233,0,1695.45,...,39.39,13.35,80.091612,0.762934,1.0,26.04,67.274026,1293.516254,Nova Mutum,treino
3,5107925,2001,390000.0,390000.0,1269450.0,3255.0,3284.0,0.991169,0,1409.06,...,39.24,15.34,81.032810,0.727645,1.0,23.90,55.588570,1025.294940,Sorriso,treino
4,5108006,2001,133555.0,133555.0,411617.0,3082.0,2853.2,1.080191,0,1522.24,...,39.11,14.84,81.698926,0.734752,1.0,24.27,60.245225,1118.468985,Tapurah,treino


## Tópico 5: Definição da Variável-Alvo
- **Alvo (`quebra_safra`)**: Definido como `1` quando o rendimento fica abaixo de `0,92 × tendência_linear`, calculada somente com o passado do próprio município (até 10 safras, mínimo 5).
- **Target Leakage**: Variáveis que representam informações de "pós-colheita" não podem ser usadas para prever o risco antes da colheita. Por isso, excluímos: `rendimento_kg_ha`, `quantidade_produzida_t`, `area_colhida_ha`, `area_plantada_ha`.

In [2]:
df_target = df_limpo.copy()

# 1. Cria o alvo atual usando tendência municipal calculada só com o passado
ano_col = "safra" if "safra" in df_target.columns else "ano"
municipio_col = "municipio" if "municipio" in df_target.columns else None

quebras = []
for idx, row in (df_target.sort_values([municipio_col, ano_col]).iterrows() if municipio_col else df_target.sort_values(ano_col).iterrows()):
    ano = int(row[ano_col])
    historico = df_target
    if municipio_col:
        historico = historico[historico[municipio_col] == row[municipio_col]]
    passado = historico[(historico[ano_col] < ano)].tail(10)
    rendimento = float(row["rendimento_kg_ha"])
    if len(passado) < 5:
        quebras.append(0)
    else:
        x = passado[ano_col].astype(float).values
        y = passado["rendimento_kg_ha"].astype(float).values
        coef = np.polyfit(x, y, 1)
        tendencia = coef[0] * ano + coef[1]
        quebras.append(int(rendimento < 0.92 * tendencia))

df_target["quebra_safra"] = 0
for pos, idx in enumerate(df_target.sort_values([municipio_col, ano_col]).index if municipio_col else df_target.sort_values(ano_col).index):
    df_target.loc[idx, "quebra_safra"] = quebras[pos]

# 2. Remoção das variáveis vazadas (Target Leakage)
colunas_vazamento = [
    "rendimento_kg_ha", 
    "quantidade_produzida_t", 
    "area_colhida_ha", 
    "area_plantada_ha"
]
df_target = df_target.drop(columns=colunas_vazamento)

print("Colunas após o Tópico 5:")
print(df_target.columns.tolist())
display(df_target[[ano_col, "quebra_safra"]].head())

print("\nDistribuição da variável-alvo (quebra_safra):")
print(df_target["quebra_safra"].value_counts())
print("\nPercentual:")
print(df_target["quebra_safra"].value_counts(normalize=True) * 100)

Colunas após o Tópico 5:
['cod_municipio', 'safra', 'rend_ref', 'razao_rend_ref', 'quebra_safra', 'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT', 'cobertura_clima_min', 'amplitude_termica', 'precipitacao_por_temperatura', 'GWET_x_PREC', 'municipio', 'split']


,safra,quebra_safra
0,2001,0
1,2001,0
2,2001,0
3,2001,0
4,2001,0



Distribuição da variável-alvo (quebra_safra):
quebra_safra
0    166
1     26
Name: count, dtype: int64

Percentual:
quebra_safra
0    86.458333
1    13.541667
Name: proportion, dtype: float64


## Tópico 6: Criação de Novas Variáveis (Feature Engineering)
Vamos criar variáveis meteorológicas derivadas que possam explicar melhor o estresse sofrido pela planta:
1. **Amplitude Térmica (`amplitude_termica`)**: Variações muito grandes de temperatura no mesmo dia geram estresse hídrico e fisiológico.
2. **Índice de Precipitação por Temperatura (`precipitacao_por_temperatura`)**: Mede o balanço de água disponível frente à evaporação gerada pelo calor.
3. **Interação Umidade x Precipitação (`GWET_x_PREC`)**: Identifica anos onde choveu muito e o solo efetivamente reteve essa água.

In [3]:
df_features = df_target.copy()

# 1. Amplitude Térmica Diária Média
df_features["amplitude_termica"] = df_features["T2M_MAX"] - df_features["T2M_MIN"]

# 2. Indice de Estresse Hídrico Simples (Chuva / Temperatura Média)
df_features["precipitacao_por_temperatura"] = df_features["PRECTOTCORR"] / (df_features["T2M"] + 0.001)

# 3. Interação: Umidade do Solo x Precipitação
df_features["GWET_x_PREC"] = df_features["GWETROOT"] * df_features["PRECTOTCORR"]

display(df_features.head())

,cod_municipio,safra,rend_ref,razao_rend_ref,quebra_safra,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,GWETROOT,cobertura_clima_min,amplitude_termica,precipitacao_por_temperatura,GWET_x_PREC,municipio,split
0,5105259,2001,3303.4,0.963553,0,1532.44,25.575992,39.67,13.98,78.689835,0.746281,1.0,25.69,59.914787,1143.630843,Lucas do Rio Verde,treino
1,5105903,2001,2740.1,0.875880,0,1419.54,25.904628,38.50,14.06,73.274752,0.395579,1.0,24.44,54.796587,561.539521,Nobres,treino
2,5106224,2001,2801.6,1.092233,0,1695.45,25.202149,39.39,13.35,80.091612,0.762934,1.0,26.04,67.271356,1293.516254,Nova Mutum,treino
3,5107925,2001,3284.0,0.991169,0,1409.06,25.348017,39.24,15.34,81.032810,0.727645,1.0,23.90,55.586377,1025.294940,Sorriso,treino
4,5108006,2001,2853.2,1.080191,0,1522.24,25.267397,39.11,14.84,81.698926,0.734752,1.0,24.27,60.242841,1118.468985,Tapurah,treino


## Salvando os dados processados
Salvaremos na pasta `data/interim/` para que os próximos passos do projeto possam consumi-los.

In [4]:
# Criar diretório se não existir
INTERIM.mkdir(parents=True, exist_ok=True)

# Salvar o dataset com features
path_out = INTERIM / "04_dados_com_features.csv"
df_features.to_csv(path_out, index=False)
print(f"Dados salvos com sucesso em: {path_out}")
print(f"Total de registros: {len(df_features)}")

Dados salvos com sucesso em: /home/jean/Documentos/bx_produtividade_agricola/data/interim/04_dados_com_features.csv
Total de registros: 192
